In [172]:
from FlagEmbedding import BGEM3FlagModel
from FlagEmbedding import FlagReranker
import numpy as np
from pathlib import Path
from tqdm.notebook import tqdm
from pymongo import MongoClient, UpdateOne

Carregando o modelo BGE-M3

In [3]:
model = BGEM3FlagModel('BAAI/bge-m3', use_fp16=True)

Loading weights: 100%|██████████| 391/391 [00:00<00:00, 468.61it/s]


Carregando os dados dos professores do Banco de dados do MongoDB

In [27]:
client = MongoClient("mongodb://localhost:27017/")
db = client["TCC"]
professores = db["Docentes"]

perfis = list(professores.find({}))

Criação de texto somente com o essencial das informações dos professoes (por enquanto só resumo + disciplinas)

In [61]:
def criar_perfil(prof):
    
    nomes_disciplinas = [d["nome"] for d in prof.get("disciplinasSigaa", [])]
    disciplinas_text = ", ".join(nomes_disciplinas)
    
    perfil = f"Resumo : {prof.get('resumo', '')} Disciplinas: {disciplinas_text}"
        
    return perfil.strip()

In [63]:
professores_texto = []
for professor in perfis:
    perfil_prof = criar_perfil(professor)
    professores_texto.append(perfil_prof)

Realizar o embedding e peso lexical de todo o texto

In [110]:
text_transformed = model.encode(professores_texto, batch_size=12, max_length=512, return_dense=True, return_sparse=True)

Inference Embeddings: 100%|██████████| 7/7 [03:12<00:00, 27.49s/it]


In [ ]:
embeddings_professores = text_transformed['dense_vecs']
lexical_professores = text_transformed["lexical_weights"]

Adicionando o embedding ao MongoDB

In [ ]:
professores_validos = list(professores.find({"embedding_dense": {"$exists": False}}))
operations = []

for perfil, dense, sparse in zip(professores_validos, embeddings_professores, lexical_professores):
    sparse_clean = {str(k): float(v) for k, v in sparse.items()}
    
    operations.append(
        UpdateOne(
            {"_id": perfil["_id"]},
            {"$set": {
                "embedding_dense": dense.tolist(),
                "embedding_sparse": sparse_clean,
            }}
        )
    )
    professores.bulk_write(operations)

Simulação da comparação com um aluno

In [202]:
aluno_texto = "Compiladores"
aluno = model.encode(aluno_texto, batch_size=12, max_length=512, return_dense=True, return_sparse=True)
embedding_aluno = aluno['dense_vecs']
lexical_aluno = aluno['lexical_weights']


In [203]:
perfis = list(professores.find({"embedding_dense": {"$exists": True}}))

In [204]:
results = []
for professor in perfis:
    
    # Denso
    dense_professor = np.array(professor["embedding_dense"])
    dense_score = embedding_aluno @ dense_professor
    
    # Esparso
    teacher_sparse = {k: v for k, v in professor["embedding_sparse"].items()}
    sparse_score = model.compute_lexical_matching_score(lexical_aluno, teacher_sparse) * 10
    
    hybrid_score = 0.6 * dense_score + 0.4 * sparse_score
    # normalizar o sparse porque está muito baixo
    results.append((professor, dense_score, sparse_score, hybrid_score))
    

In [205]:
ranked = sorted(results, key=lambda x: x[3], reverse=True)

for prof, dense_s, sparse_s, hybrid_s in ranked[:5]:
    print(f"{prof['nome']} — dense: {dense_s:.4f}, sparse: {sparse_s:.4f}, hybrid: {hybrid_s:.4f}")

Thatyana de Faria Piola Seraphim — dense: 0.4776, sparse: 0.3848, hybrid: 0.4405
Maurilio Pereira Coutinho — dense: 0.3954, sparse: 0.1773, hybrid: 0.3081
Enzo Seraphim — dense: 0.4456, sparse: 0.0000, hybrid: 0.2673
Bruno Tardiole Kuehne — dense: 0.4251, sparse: 0.0000, hybrid: 0.2551
Luciano Bertini — dense: 0.3704, sparse: 0.0793, hybrid: 0.2540


Realizando o reranking com os Top 8 professores

In [200]:
reranker = FlagReranker('BAAI/bge-reranker-v2-m3', use_fp16=True, normalize=True)

Loading weights: 100%|██████████| 393/393 [00:00<00:00, 5614.27it/s]


In [206]:
pairs = []
for prof, dense_s, sparse_s, hybrid_s in ranked[:8]:
    professor_texto = criar_perfil(prof)
    pairs.append([aluno_texto, professor_texto])

rerank_scores = reranker.compute_score(pairs)

In [184]:
final_ranked = sorted(
    zip([c[0] for c in ranked[:8]], rerank_scores),
    key=lambda x: x[1],
    reverse=True
)

In [185]:
for prof, hybrid in final_ranked[:5]:
    print(f"{prof['nome']} — rerank score: {hybrid:.4f}")

Thatyana de Faria Piola Seraphim — rerank score: 0.3017
Edmilson Marmo Moreira — rerank score: 0.0054
João Paulo Reus Rodrigues Leite — rerank score: 0.0052
Bruno Tardiole Kuehne — rerank score: 0.0040
Edvard Martins de Oliveira — rerank score: 0.0039


Adicionar 

- Normalizar os valores (Lexical)
- Fine-tuning
- Avaliar quais informações são relevantes de se obter dos professores
- Que dados serão obtidos dos alunos